# Using and Building a Prediction Market

## Introduction

#### The Core Concept: Information Aggregation
*   **Definition:** Marketplaces where participants trade shares of future event outcomes. The price of a share directly reflects the market's perceived probability of that outcome.
*   **How it Works:** Financial incentives drive accuracy. If a trader believes the current price is wrong, they buy underpriced shares or sell overpriced ones to profit. This continuous trading automatically aggregates disparate private knowledge into a single, accurate collective forecast.

#### Eliciting Truth: From Scoring Rules to Markets
*   **Proper Scoring Rules (Traditional):** Mathematical formulas (e.g., Brier score, Log score) designed to reward forecasters based on accuracy. They ensure that a forecaster's most profitable strategy is to report their true belief. 
    *   *Limitation:* They are "one-shot" mechanisms, making it difficult to aggregate multiple people's opinions simultaneously.
*   **Market Scoring Rules:** Introduced by Robin Hanson, this mechanism transitions scoring to a sequential market setting. Each trader effectively corrects the previous trader's prediction, updating the probability and establishing a new consensus.

---

In this lab, you will explore prediction markets both as a **participant** and as a **builder**:
- **Part 1: Using Linera Markets** – You will participate in a live prediction market on the Linera platform, placing bets on the outcome of a random event. This hands-on experience will help you understand how market prices evolve based on trading activity and how they reflect collective beliefs.
- **Part 2: AMM Calculations** – You will solve a few short problems to solidify your understanding of how automated market makers (like LMSR and Uniswap’s constant product AMM) calculate prices and respond to trades.
- **Part 3: Coding an LMSR Market Maker** – You will examine a simplified Solidity smart contract that implements an LMSR-based prediction market for a binary outcome, with comments explaining how the mechanism works.

Let's get started!

---
---


## Part 1: Participating in a Prediction Market

- Linera Market: https://linera.market/?market=BTC&duration=1 

### How does the prediction market look like?
- Polymarket: https://polymarket.com/ 
- How does it work: https://docs.polymarket.com/ 
- A real-money prediction market platform built on Ethereum mainnet. It offers a wide range of markets on various topics, including politics, sports, and current events. Users can trade in USDC to buy shares of different outcomes, and the market prices reflect the collective belief of the participants.

‼️ Note: Polymarket operates on the Ethereum mainnet and requires real money (USDC) to participate. For this lab, we will use the Linera Market on the Somnia testnet, which allows you to participate without risking real funds.


### Participating Instructions (for the Linera Market)
- Linera Market: https://linera.market/?market=BTC&duration=1
- Repo: https://github.com/linera-io/linera-protocol 

Step 1: Connect & Claim Test Tokens
* Go to the Linera Market platform and connect your wallet.
* Upon registration, you will automatically receive **200 GMIC** (the platform's test currency) to start trading risk-free. 

Step 2: Select Your Market & Timeframe
* Choose an underlying asset (e.g., **ETH**, **BTC**, or **SOL**).
* Select a highly compressed timeframe (e.g., **1 Minute**, **3 Minutes**, or **5 Minutes**).

Step 3: Make Your Prediction
* Look at the current real-time price.
* Decide where the price will be at the end of the countdown:
  * Click **"HIGHER"** if you think the price will go up.
  * Click **"LOWER"** if you think the price will drop.
* Enter the amount of GMIC you want to stake and confirm the transaction.

Step 4: Wait for Instant Resolution
* Once the countdown hits zero, the smart contract fetches the final price via an oracle.
* If you guessed correctly, your GMIC balance increases. If wrong, your staked GMIC is lost.


**Parimutuel**

1. **The Pools:** All GMIC staked on "HIGHER" goes into **Pool A**. All GMIC staked on "LOWER" goes into **Pool B**. 
2. **The Total Pot:** Let's say Pool A has 400 GMIC, and Pool B has 600 GMIC. The Total Pool is 1,000 GMIC.
3. **The Outcome:** The 1-minute timer ends, and ETH price went *UP*.
4. **The Payout:** The players in Pool B (LOWER) lose their money. The 600 GMIC from Pool B is distributed to the winners in Pool A, **proportionally based on their original stake**. 

----

## Part 2: AMM Calculation Exercises
Next, let's reinforce some concepts with a couple of calculation exercises. These will help you understand quantitatively how market makers set prices in prediction markets.

### Exercise 1: LMSR (Logarithmic Market Scoring Rule) Price Impact

**Background:**
LMSR is the foundational algorithm for prediction markets (used by early Polymarket and Manifold Markets). Its core concept is that the market's "total liquidity pool" is governed by a cost function $C(q_{yes}, q_{no})$, where $q_{yes}$ and $q_{no}$ are the number of shares for each outcome in the market. The cost to buy shares is simply the **difference in this cost function** before and after the trade. 

The LMSR cost function for two outcomes is 

$$C(q_{yes}, q_{no}) = b \cdot \ln\!\Big(e^{q_{yes}/b} + e^{q_{no}/b}\Big),$$

**Given:**
*   Liquidity parameter $b = 10$. Higher $b$ means more liquidity. 
*   Initial state: $q_{yes} = 0, q_{no} = 0$, which corresponds to a 50% implied probability for both outcomes.
*   After-trade state: $q_{yes} = 5, q_{no} = 0$
*   Math hints: $e^{0.5} \approx 1.6487$, $\ln 2 \approx 0.6931$, $\ln(2.6487) \approx 0.9741$

**Calculate the cost for the trader** TODO



*(Conceptually: Even though you are buying 5 shares, the price dynamically increases as you buy. Therefore, the average cost per share is about 0.56 instead of 0.5, since the price moves up with each share purchased.)*

**Calculate the new implied probability of "Yes"** TODO



### Exercise 2: Constant-Product AMM

**Background:**
This is the most widely used model in DeFi (like Uniswap), governed by the invariant formula $X \times Y = K$ (where K is a constant). In a prediction market, the liquidity pool holds tokens representing both outcomes. **If you want to take one type of token out of the pool, you must put the other type in to maintain the curve's invariant.**

**Given:**
*   Initial token reserves: $X_{yes} = 50, X_{no} = 50$
*   The invariant (Constant $K$): $X_{yes} \times X_{no} = 2500$
*   Trade action: The trader **takes (buys)** 10 "Yes" tokens from the pool.

**Calculate how many "No" tokens the trader must put into the pool** TODO

**Calculate the new pool reserves and new implied probability** TODO




## Part 3: Implementing an LMSR Market Maker in Solidity

Finally, let's examine how we could implement a prediction market as a smart contract. Below is a simplified Solidity contract for a binary outcome prediction market using the LMSR mechanism. The contract acts as an automated market maker:
- It allows users to buy "Yes" or "No" outcome shares by paying Ether (which acts as the currency).
- It keeps track of the total shares and uses the LMSR formula to determine the cost of shares (and thus the price).
- When the outcome is resolved (set by the owner of the contract), holders of the winning outcome shares can redeem 1 Ether per share (losing shares become worthless).

For simplicity, this contract uses Ether as the currency and does not use an external price oracle for resolution (we assume the owner will call the resolve function with the true outcome). In a real-world scenario, you'd want a decentralized way to resolve the market (for example, using a trusted oracle or a voting mechanism).

⚠️ **Note:** Implementing the LMSR cost function involves using exponentials and logarithms, which Solidity can't do with native types directly (at least not with full precision). In this code, the cost calculation is shown conceptually. In practice, you'd use a fixed-point math library or iterative approximation to calculate the cost and price. The focus here is on the structure and logic rather than the exact math implementation. We use the ABDK libraries to do this calculation.


```javascript
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

// Import the ABDK Math library for fixed-point math functions (ln, exp, etc.)
import "abdk-libraries-solidity/ABDKMath64x64.sol";

contract LMSRPredictionMarket {
    using ABDKMath64x64 for int128;

    // Liquidity parameter for LMSR: higher = more liquidity, less price movement per trade
    uint256 public immutable b; // fixed liquidity parameter
    address public owner; // market creator who can resolve the market
    bool public marketResolved; // whether the market has been resolved
    bool public outcome; // true = Yes wins, false = No wins

    // Totals of shares purchased
    uint256 public totalYesShares; // q_Yes in the LMSR formula
    uint256 public totalNoShares; // q_No in the LMSR formula

    // Each user's Yes/No share balance
    mapping(address => uint256) public yesBalance;
    mapping(address => uint256) public noBalance;

    constructor(uint256 _b) payable {
        b = _b;
        owner = msg.sender;
    }

    modifier onlyOwner() {
        require(msg.sender == owner, "Not owner");
        _;
    }

    // Calculate LMSR cost for a trade using exponential scoring rule
    function calculateCost(
        uint256 qYesBefore, // total Yes shares before the trade
        uint256 qNoBefore, // total No shares before the trade
        uint256 qYesAfter, // total Yes shares after the trade
        uint256 qNoAfter // total No shares after the trade
    ) public view returns (uint256) {
        int128 b64 = ABDKMath64x64.fromUInt(b);

        // Convert quantities to fixed-point format and divide by b (q/b)
        int128 qYes1 = ABDKMath64x64.fromUInt(qYesBefore).div(b64);
        int128 qNo1 = ABDKMath64x64.fromUInt(qNoBefore).div(b64);
        int128 qYes2 = ABDKMath64x64.fromUInt(qYesAfter).div(b64);
        int128 qNo2 = ABDKMath64x64.fromUInt(qNoAfter).div(b64);

        // Compute cost before and after the trade 
        // (C(q_Yes, q_No) = b * ln(e^(q_Yes/b) + e^(q_No/b)))
        int128 cost1 = (qYes1.exp().add(qNo1.exp())).ln();
        int128 cost2 = (qYes2.exp().add(qNo2.exp())).ln();

        // Final cost = b * (cost2 - cost1)
        int128 costDelta = (cost2.sub(cost1)).mul(b64);

        return ABDKMath64x64.mulu(costDelta, 0.01 ether); // 1 share = 0.01 Ether
    }

    // Buy Yes shares and pay Ether based on cost function
    function buyYes(uint256 amount) external payable {
        require(!marketResolved, "Market resolved");

        // Calculate cost of buying additional Yes shares
        uint256 cost = calculateCost(
            totalYesShares, 
            totalNoShares, 
            totalYesShares + amount,  // new total Yes shares after the trade
            totalNoShares
        );

        require(msg.value >= cost, "Insufficient payment");

        // TODO: Update user balance and total shares

        // TODO: Refund any excess ETH sent

        
    }

    // Buy No shares and pay Ether based on cost function
    function buyNo(uint256 amount) external payable {
        require(!marketResolved, "Market resolved");

        // Calculate cost of buying additional No shares
        uint256 cost = calculateCost(
            // TODO: Fill in the correct parameters for the cost calculation

        );

        require(msg.value >= cost, "Insufficient payment");

        // TODO: Update user balance and total shares
        
        // TODO: Refund any excess ETH sent

    }

    // Resolve the market with the actual outcome (only callable by owner)
    function resolveMarket(bool _outcome) external onlyOwner {
        require(!marketResolved, "Already resolved");
        marketResolved = true;
        outcome = _outcome;
    }

    // Redeem winning shares for ETH (1 share = 1 ETH payout)
    function redeem() external {
        require(marketResolved, "Market not resolved");

        if (outcome) {
            // TODO: If "Yes" wins


        } else {
            // TODO: If "No" wins

        }
    }
}

```

**Contract deployed on Sepolia testnet:**
https://sepolia.etherscan.io/address/0xda26F8CA3eC7653C90763b1e17B6aeB4Bcde2063#readContract

**Instructions to interact with the contract:**
1. Go to the "Write Contract" tab and buy some "Yes" or "No" shares by calling the `buyYes` or `buyNo` function with an appropriate amount and sending enough Ether to cover the cost.
2. You can calculate the cost of your trade using the `calculateCost` function by inputting the current total shares and the new total shares after your intended purchase. This will help you understand how much Ether you need to send for your trade.
3. If every student in the class buys some shares, GTA will resolve the market. 
4. Then, you can call the `redeem` function to claim your winnings if you hold shares of the winning outcome.
5. Your profit will be the number of winning shares you hold multiplied by 0.01 Ether minus the cost you paid to buy those shares.

As the number of participants who buy "Yes" increases, the price of "Yes" shares will go up, reflecting the market's belief that "Yes" is more likely to be the correct outcome.
